In [ ]:
# config.py
import torch
import os
import matplotlib.pyplot as plt
from PIL import Image

# --- Training Hyperparameters ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE_GEN = 1e-4
LEARNING_RATE_DISC = 1e-4
BATCH_SIZE = 16
NUM_EPOCHS_PRETRAIN = 50
NUM_EPOCHS_GAN = 100
HIGH_RES_SIZE = 96  # As per the paper
LOW_RES_SIZE = HIGH_RES_SIZE // 4
NUM_WORKERS = 4
LAMBDA_VGG = 1.0  # Weight for VGG/content loss
LAMBDA_ADV = 1e-3  # Weight for adversarial loss

os.makedirs("saved_models", exist_ok=True)

# --- Model Paths ---
PRETRAINED_GEN_PATH = "saved_models/srresnet_pretrained.pth"
GEN_PATH = "saved_models/generator.pth"
DISC_PATH = "saved_models/discriminator.pth"

# --- Dataset Paths ---
TRAIN_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_train_HR"
TEST_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_valid_HR"

# set the repo name
model_name = "keanteng/srgan-div2k-0723-v2"

## From The Paper



In [ ]:
# dataset.py
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class ImageDataset(Dataset):
    """
    Custom dataset to load high-resolution images and create low-resolution counterparts.
    """
    def __init__(self, hr_dir, hr_size):
        super(ImageDataset, self).__init__()
        self.hr_image_files = [os.path.join(hr_dir, f) for f in os.listdir(hr_dir)]
        self.hr_size = hr_size

        # Transform for the original image before cropping
        self.initial_transform = transforms.Compose([
            transforms.ToTensor(), # lr image go through here will be scaled to [0, 1]
        ])

        # Normalization transforms
        self.hr_normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # normalize to [-1, 1]
        # take 0 and 1 range
        # 0: (0-0.5)/0.5 = -1
        # 1: (1-0.5)/0.5 = 1
        self.lr_normalize = transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0]) # nothing change x = (x - mean) / std, so if mean=0 and std=1, x remains unchanged, we will use tanh to scale to [-1,1] in the generator

    # called by pytorch DataLoader
    # get a single item for each dataset using the index
    def __getitem__(self, index):
        # Load image
        hr_image = Image.open(self.hr_image_files[index]).convert("RGB")

        # Convert to tensor first
        hr_tensor = self.initial_transform(hr_image)

        # Apply random crop to get consistent size
        crop_transform = transforms.RandomCrop(self.hr_size)
        hr_cropped = crop_transform(hr_tensor)

        # Create LR version by downsampling the cropped HR image
        lr_tensor = transforms.functional.resize(
            hr_cropped,
            size=self.hr_size // 4,
            interpolation=transforms.InterpolationMode.BICUBIC # when u upscale, fill the neighboring pixel values
            # when downscale, it will use the average of the neighboring pixel values
        )

        # Apply normalization
        hr_normalized = self.hr_normalize(hr_cropped)
        lr_normalized = self.lr_normalize(lr_tensor)

        return lr_normalized, hr_normalized

    # called by pytorch DataLoader
    # return the length of the dataset
    # SO HOW MANY IMAGES ARE THERE IN THE DATASET?
    def __len__(self):
        return len(self.hr_image_files)

In [ ]:
dataset = ImageDataset(hr_dir=TRAIN_DIR, hr_size=HIGH_RES_SIZE)